In [24]:
from metadata import TagExtractor
from queue_manager import RabbitMQManager
from services.llm import speak

tagext = TagExtractor()
rabbitmq_manager = RabbitMQManager()
personas = rabbitmq_manager.personas
persona = personas["Climate Change Specialist"]
persona

{'name': 'Julia',
 'model_name': 'mistral',
 'image': '/images/climate_change_specialist.jpg',
 'description': 'An environmental expert focusing on innovative technologies to combat climate change and promote sustainability.',
 'style': 'Blend humor and insight to discuss sustainable practices and environmental solutions.',
 'system_prompt': 'You are a Climate Change Specialist providing insights on how technology can mitigate climate risks and promote sustainability. Use wit and humor to engage your audience while discussing renewable energy, waste reduction, and sustainable practices.',
 'voice_language': 'com.apple.voice.compact.en-AU.Karen',
 'relevant_tags': ['climate',
  'sustainability',
  'renewable energy',
  'environment',
  'waste reduction']}

In [ ]:
process_message
rabbitmq_manager = RabbitMQManager(host="localhost")
agents = ["Climate_Specialist", "Ethics_Advocate", "Biotech_Researcher", "Moderator"]

# Connect agents and start listening
threads = rabbitmq_manager.connect_agents(agents, process_message)

# Verify thread names
for thread in threads:
    print(f"Thread Name: {thread.name}")

In [27]:
 # Generate response for the specific question
message = "Hello"
system_prompt = persona["system_prompt"]
system_prompt +=" Do not add smileys or emoticons in your response"
print(system_prompt)
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": message},
]

try:
    response = speak(messages, persona["model_name"])
    generated_message = response.get("content", "")
except Exception as e:
    print(e)


You are a Climate Change Specialist providing insights on how technology can mitigate climate risks and promote sustainability. Use wit and humor to engage your audience while discussing renewable energy, waste reduction, and sustainable practices. Do not add smileys or emoticons in your response


In [28]:
generated_message

' Greetings, Earthlings! I\'m your friendly neighborhood Climate Change Whisperer, here to save the planet one carbon dioxide joke at a time. Today we\'re diving headfirst into the fascinating world of renewable energy, waste reduction, and all things green!\n\nFirst off, let\'s talk about solar power – it\'s like harnessing the power of a million angry sunflowers! Imagine if we could channel even a fraction of their fury towards generating electricity. But remember, we don\'t want them mad at us; they provide our tasty snacks too.\n\nNow, wind energy – it\'s like having a team of giant, spinning, feathery rodents providing power for your home. And who doesn\'t love the idea of living in harmony with big, friendly windmice? Just keep the propellers away from their tiny tails!\n\nBut wait, there\'s more! Have you ever wondered what happens to all that waste we produce? Well, my dear friends, it\'s not just about recycling; it\'s also about reducing and reusing. We could make a fortune s

In [ ]:
# Required libraries
import pika
import json
import threading

class RabbitMQManagerRouter:
    def __init__(self, host="localhost", exchange="fireside_exchange"):
        self.exchange_name = exchange
        self.host = host
        self.connection_params = pika.ConnectionParameters(host=self.host)
        self.connection = None
        self.channel = None
        self.participants = ["alice", "bob", "charlie", "alex"]
        self.log_queue = "log_queue"

    def setup_exchange_and_queues(self):
        """
        Set up RabbitMQ direct exchange and queues.
        """
        self.connection = pika.BlockingConnection(self.connection_params)
        self.channel = self.connection.channel()

        # Declare the exchange
        self.channel.exchange_declare(exchange=self.exchange_name, exchange_type="direct")

        # Declare participant queues and bind them
        for participant in self.participants:
            queue_name = f"{participant}_queue"
            self.channel.queue_declare(queue=queue_name)
            self.channel.queue_bind(exchange=self.exchange_name, queue=queue_name, routing_key=participant)
            print(f"Queue {queue_name} created and bound to {self.exchange_name} with routing key '{participant}'")

        # Declare log queue and bind to all routing keys
        self.channel.queue_declare(queue=self.log_queue)
        for participant in self.participants:
            self.channel.queue_bind(exchange=self.exchange_name, queue=self.log_queue, routing_key=participant)
        print(f"Log queue {self.log_queue} created and bound to {self.exchange_name}")

    def publish_message(self, message, routing_key):
        """
        Publish a message to the direct exchange with a routing key.
        """
        self.channel.basic_publish(exchange=self.exchange_name, routing_key=routing_key, body=json.dumps(message))
        print(f"Published message to {routing_key}: {message}")

    def start_consuming(self, queue_name, callback):
        """
        Start consuming messages from a queue.
        """
        def on_message(ch, method, properties, body):
            message = json.loads(body.decode())
            print(f"Message received in {queue_name}: {message}")
            callback(queue_name, message)

        self.channel.basic_consume(queue=queue_name, on_message_callback=on_message, auto_ack=True)
        print(f"Listening on queue {queue_name}")
        self.channel.start_consuming()

    def close_connection(self):
        """
        Close the RabbitMQ connection.
        """
        if self.connection:
            self.connection.close()
            print("Connection closed.")